# Mass generation — seen generator B (Mistral via OpenRouter)

Генерирует **~40 %** той же seen-сетки, что и ноутбук 11: при **`OPENAI_SEEN_SHARE=0.6`** слоты Mistral — дополнение к OpenAI. Пересечения `generation_job_id` между 11 и 12 **нет** (один слот → ровно один генератор).

- **API:** `https://openrouter.ai/api/v1`, ключ **`OPENROUTER_API_KEY_MISTRAL`**.
- **Output:** `v2/data/interim/llm-generated/core_llm_seen_mistral_<model_slug>.jsonl`.
- **Split:** `seen` — только train/val при сборке.

**Сетка:** те же **`SAMPLES_PER_SUBTYPE`** и **`OPENAI_SEEN_SHARE`**, что в `11_mass_generation_openai.ipynb` (или общий `OPENAI_SEEN_SHARE` в `.env`).

**Env:** `OPENROUTER_API_KEY_MISTRAL` в `v2/.env` или корневом `.env`. Ключ Claude — только **`OPENROUTER_API_KEY_CLAUDE`** (ноутбук 13).

`v2/docs/dataset_design_final.md` §6.2, §7.2.

**Скорость:** `LLM_GEN_MAX_WORKERS` (как в ноутбуке 11) + при необходимости запуск **11 и 12 параллельно**.

**Прогресс:** `tqdm` + postfix (см. `v2/src/llm_mass_generation.py`).

In [3]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

V2 = None
cur = Path.cwd().resolve()
for _ in range(24):
    if (cur / "config.py").is_file() and (cur / "src" / "llm_mass_generation.py").is_file():
        V2 = cur
        break
    nested = cur / "v2"
    if (nested / "config.py").is_file() and (nested / "src" / "llm_mass_generation.py").is_file():
        V2 = nested
        break
    if cur.parent == cur:
        break
    cur = cur.parent
if V2 is None:
    raise RuntimeError(
        "Не найден v2/: открой папку репозитория или v2/, либо смени cwd ядра Jupyter."
    )

sys.path.insert(0, str(V2))

from src.llm_mass_generation import MassGenConfig, run_mass_generation

load_dotenv(V2.parent / ".env")
load_dotenv(V2 / ".env")

# Must match notebook 11 (production: 400 × 23 = 9200 seen slots total).
SAMPLES_PER_SUBTYPE = 400
OPENAI_SEEN_SHARE = float(os.environ.get("OPENAI_SEEN_SHARE", "0.6"))
MAX_WORKERS = max(1, int(os.environ.get("LLM_GEN_MAX_WORKERS", "8")))

# OpenRouter model id for Mistral — see https://openrouter.ai/models
MODEL = "mistralai/mistral-small-3.1-24b-instruct"

cfg = MassGenConfig(
    base=V2,
    lane="seen_mistral",
    api_key_env="OPENROUTER_API_KEY_MISTRAL",
    model=MODEL,
    origin_model=f"openrouter/{MODEL}",
    split="seen",
    samples_per_subtype=SAMPLES_PER_SUBTYPE,
    openai_base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "https://local.invalid/v2-core-llm",
        "X-Title": "v2 Core LLM seen (Mistral via OpenRouter)",
    },
    openai_seen_share=OPENAI_SEEN_SHARE,
    max_workers=MAX_WORKERS,
)

run_mass_generation(cfg)

Output:        /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_seen_mistral_mistralai_mistral-small-3_1-24b-instruct.jsonl
Lane:          seen_mistral
Model:         mistralai/mistral-small-3.1-24b-instruct
origin_model:  openrouter/mistralai/mistral-small-3.1-24b-instruct
split:         seen
openai_seen_share (seen split): 0.6
max_workers:   8
Jobs total:    3688
Already done:  3688
Pending:       0


seen_mistral: |          | 0/0 [00:00<?, ?job/s] 

Done. In file: 3688/3688 rows. Path: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_seen_mistral_mistralai_mistral-small-3_1-24b-instruct.jsonl


PosixPath('/Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_seen_mistral_mistralai_mistral-small-3_1-24b-instruct.jsonl')